In [1]:
import sys, os
import torch

MODEL_ROOT = "/home/dsail/2026-AINews-Detection/model"
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)
os.chdir(MODEL_ROOT)

from eval import load_detector, patch_parse_args, setup_cuda_fast

# ── 설정 ────────────────────────────────────────────────────────
CHECKPOINT = "/home/dsail/2026-AINews-Detection/model/results/2026-06-14/kor_aigen-20-58-32-752539/kor_aigen_30_8_0.7_20260614-205835.pth"
INFER_STRUCTURE_WEIGHT = 0.7   # 1=structure only, 0=unit only
POS_BIT = 0                 # eval_newsgpt.py 기본값 (0=AI score index)
NO_AMP  = False

# ── args 패치 (eval_newsgpt.py와 동일) ──────────────────────────
patch_parse_args(infer_structure_weight=INFER_STRUCTURE_WEIGHT, no_amp=NO_AMP)

# ── 모델 로드 ────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
run_dir = os.path.dirname(CHECKPOINT)
model = load_detector(run_dir, DEVICE, POS_BIT, CHECKPOINT)
model.eval()
setup_cuda_fast(DEVICE)

print(f"device                 : {DEVICE}")
print(f"checkpoint             : {os.path.basename(CHECKPOINT)}")
print(f"infer_structure_weight : {INFER_STRUCTURE_WEIGHT}")
print("모델 로드 완료")

/home/dsail/anaconda3/envs/aigen_det/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device                 : cuda
checkpoint             : kor_aigen_30_8_0.7_20260614-205835.pth
infer_structure_weight : 0.7
모델 로드 완료


In [2]:
text = "중소·중견기업의 글로벌 수출 경쟁력 강화를 위한 새로운 금융 지원이 시작된다. 한국무역보험공사는 하나은행, 우리은행과 함께 9350억원 규모의 ‘수출패키지 우대금융’을 신설한다고 3일 밝혔다. 이를통해 중소·중견기업의 자금 애로 해소와 이자 부담을 줄여주겠다는 것이다.\n무보는 이날 하나은행, 우리은행과 업무협약을 체결했다고 발표했다. 이 협약은 지난해 12월 산업통상자원부 장관이 참석한 가운데 5대 시중은행과의 포괄적 업무협약에 이은 후속 조치다. 협약의 주요 내용은 수출에 필요한 자금을 저비용으로 지원하는 것이다. 이를 통해 중소·중견기업이 글로벌 시장에서 경쟁력을 높일 수 있을 것으로 기대된다고 무보는 밝혔다.\n무보는 은행이 추천한 기업을 대상으로 수출신용보증의 지원 한도와 보증 비율을 확대한다. 또한 보험·보증료 할인을 지원한다. 그러면 은행은 700억원 상당의 기금을 출연하고 대출 금리·수수료·환가료율을 우대한다. 이번 협약으로 중소·중견기업은 수출에 필요한 자금을 필요한 때에 지원받을 수 있게 됐다.\n무보 관계자는 “이번 협약을 통해 유동성이 필요한 기업이 도움을 받을것으로 본다”며 “금융기관과의 정책 공조를 통해 공공 부문 중심의 현행 지원구조를 민간 부문으로 확대하는 계기가 될 것으로 예상한다”고 했다.\n무보는 올해 내로 나머지 시중 은행과의 협약도 체결할 계획이다. 장진욱 무보 중소중견사업 본부장은 “고물가·고환율·고금리 현상으로 대외 수출 환경이 어려운 가운데 이번 협약이 기업들에게 실질적인 도움이 될 것”이라고 말했다.\n[조선일보와 미디어DX가 공동 개발한 생성형 AI의 도움을 받아 작성한 기사입니다.]"

with torch.inference_mode():
    probs = model.forward([text])   # shape: (1, 2) 

# eval_newsgpt.py와 동일: scores = probs[:, pos_bit]
ai_score = probs[0][POS_BIT].item()
label    = "AI" if ai_score >= 0.8 else "Human"  # 0.99

print(f"AI score (pos_bit={POS_BIT}) : {ai_score:.4f}")
print(f"판정                          : {label}")

AI score (pos_bit=0) : 0.4071
판정                          : Human


In [4]:
import json

In [ ]:
THRESHOLD = 0.997 #0.9987247586250305
newsGPT_root = "/home/dsail/2026-AINews-Detection/notebook/real-world/NewsGPT/newsgpt_ai_news_human.json"

with open(newsGPT_root, "r", encoding="utf-8") as f:
    newsgpt_data = json.load(f)

gts, preds, scores, articles = [], [], [], []

for i, data in enumerate(newsgpt_data):
    article = data["text"]
    gt = 1 if data["label"] == "llm" else 0   # "llm"=1, "human"=0

    with torch.inference_mode():
        logits = model.forward([article])   # shape: (1, 2) 

    probs = torch.softmax(logits, dim=1)
    ai_score = probs[0, POS_BIT].item()
    label = 1 if ai_score >= THRESHOLD else 0

    gts.append(gt)
    preds.append(label)
    scores.append(ai_score)
    articles.append(article)

    if (i + 1) % 50 == 0:
        print(f"  [{i+1}/{len(newsgpt_data)}] 진행 중...")

correct   = sum(g == p for g, p in zip(gts, preds))
accuracy  = correct / len(gts)

tp = sum(g == 1 and p == 1 for g, p in zip(gts, preds))
fp = sum(g == 0 and p == 1 for g, p in zip(gts, preds))
fn = sum(g == 1 and p == 0 for g, p in zip(gts, preds))
tn = sum(g == 0 and p == 0 for g, p in zip(gts, preds))

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print(f"\n{'='*40}")
print(f"총 샘플 수   : {len(gts)}")
print(f"Threshold   : {THRESHOLD}")
print(f"Accuracy    : {accuracy:.4f}  ({correct}/{len(gts)})")
print(f"Precision   : {precision:.4f}")
print(f"Recall      : {recall:.4f}")
print(f"F1 Score    : {f1:.4f}")
print(f"{'='*40}")
print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")

import pandas as pd
OUT_PATH = "/home/dsail/2026-AINews-Detection/notebook/real-world/newsgpt_predictions.csv"
df_out = pd.DataFrame({
    "text":    articles,
    "gt":      gts,
    "pred":    preds,
    "ai_score": scores,
})
df_out.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"\nCSV 저장 완료: {OUT_PATH}  ({len(df_out)}행)")

  [50/873] 진행 중...
  [100/873] 진행 중...
  [150/873] 진행 중...
  [200/873] 진행 중...
  [250/873] 진행 중...
  [300/873] 진행 중...
  [350/873] 진행 중...
  [400/873] 진행 중...
  [450/873] 진행 중...
  [500/873] 진행 중...
  [550/873] 진행 중...
  [600/873] 진행 중...
  [650/873] 진행 중...
  [700/873] 진행 중...
  [750/873] 진행 중...
  [800/873] 진행 중...
  [850/873] 진행 중...

총 샘플 수   : 873
Threshold   : 0.997
Accuracy    : 0.5155  (450/873)
Precision   : 0.5155
Recall      : 1.0000
F1 Score    : 0.6803
TP=450  FP=423  FN=0  TN=0

CSV 저장 완료: /home/dsail/2026-AINews-Detection/notebook/real-world/newsgpt_predictions.csv  (873행)


In [14]:
pred_newsgpt = pd.read_csv("/home/dsail/2026-AINews-Detection/notebook/real-world/newsgpt_predictions.csv")

misclassified = pred_newsgpt[pred_newsgpt["gt"] != pred_newsgpt["pred"]].reset_index(drop=True)
print(f"전체: {len(pred_newsgpt)}  오분류: {len(misclassified)}  정확도: {1 - len(misclassified)/len(pred_newsgpt):.4f}")
misclassified

전체: 873  오분류: 423  정확도: 0.5155


,text,gt,pred,ai_score
0,Australia Sues 3M for 2 Billion Dollars over P...,0,1,0.299367
1,Naomi Osaka Brings High Fashion to French Open...,0,1,0.302581
2,DOJ Sues UCLA over Alleged Failure to Protect ...,0,1,0.299322
3,Jill Biden Admits She Feared Joe Biden Had a S...,0,1,0.296824
4,China Executes Former Lawyer for Murder of Bil...,0,1,0.298550
...,...,...,...,...
418,David Haye Insists Joshua and Fury Must Fight ...,0,1,0.313406
419,"Manchester City Beats Arsenal, Slashes Title L...",0,1,0.308127
420,Manchester City and Arsenal in Thrilling Title...,0,1,0.303199
421,West Ham Out of Relegation Zone\n\nWest Ham’s ...,0,1,0.305884


In [ ]:
import json
import torch
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

THRESHOLD = 0.815

chosun_root = "/home/dsail/2026-AINews-Detection/notebook/real-world/Chosun/chosun_human_articles.json"

with open(chosun_root, "r", encoding="utf-8") as f:
    chosun_data = json.load(f)

gts, preds, scores, articles, titles = [], [], [], [], []

for i, data in enumerate(chosun_data):
    title = data.get("title", "")
    text = data.get("text", "")

    article = title + "\n\n" + text

    raw_label = data.get("label", 0)

    if raw_label in ["llm", "ai", "AI", "generated"]:
        gt = 1
    elif raw_label in ["human", "Human"]:
        gt = 0
    else:
        gt = int(raw_label)

    with torch.inference_mode():
        probs = model.forward([article])  # shape: (1, 2)

    ai_score = probs[0][POS_BIT].item()
    pred = 1 if ai_score >= THRESHOLD else 0

    gts.append(gt)
    preds.append(pred)
    scores.append(ai_score)
    articles.append(article)
    titles.append(title)

    if (i + 1) % 50 == 0:
        print(f"  [{i+1}/{len(chosun_data)}] 진행 중...")

accuracy = accuracy_score(gts, preds)
precision = precision_score(gts, preds, zero_division=0)
recall = recall_score(gts, preds, zero_division=0)
f1 = f1_score(gts, preds, zero_division=0)

if len(set(gts)) == 2:
    roc_auc = roc_auc_score(gts, scores)
    pr_auc = average_precision_score(gts, scores)
else:
    roc_auc = None
    pr_auc = None

tp = sum(g == 1 and p == 1 for g, p in zip(gts, preds))
fp = sum(g == 0 and p == 1 for g, p in zip(gts, preds))
fn = sum(g == 1 and p == 0 for g, p in zip(gts, preds))
tn = sum(g == 0 and p == 0 for g, p in zip(gts, preds))

print(f"\n{'='*40}")
print(f"총 샘플 수   : {len(gts)}")
print(f"Threshold   : {THRESHOLD}")
print(f"Accuracy    : {accuracy:.4f}")
print(f"Precision   : {precision:.4f}")
print(f"Recall      : {recall:.4f}")
print(f"F1 Score    : {f1:.4f}")

if roc_auc is not None:
    print(f"ROC-AUC     : {roc_auc:.4f}")
    print(f"PR-AUC      : {pr_auc:.4f}")
else:
    print("ROC-AUC     : 계산 불가 - 단일 클래스 데이터")
    print("PR-AUC      : 계산 불가 - 단일 클래스 데이터")

print(f"{'='*40}")
print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")

OUT_PATH = "/home/dsail/2026-AINews-Detection/notebook/real-world/chosun_predictions.csv"

df_out = pd.DataFrame({
    "title": titles,
    "text": articles,
    "gt": gts,
    "pred": preds,
    "ai_score": scores,
})

df_out.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print(f"\nCSV 저장 완료: {OUT_PATH} ({len(df_out)}행)")

  [50/341] 진행 중...
  [100/341] 진행 중...
  [150/341] 진행 중...
  [200/341] 진행 중...
  [250/341] 진행 중...
  [300/341] 진행 중...

총 샘플 수   : 341
Threshold   : 0.85
Accuracy    : 0.7654
Precision   : 0.8814
Recall      : 0.4160
F1 Score    : 0.5652
ROC-AUC     : 0.7610
PR-AUC      : 0.7302
TP=52  FP=7  FN=73  TN=209

CSV 저장 완료: /home/dsail/2026-AINews-Detection/notebook/real-world/chosun_predictions.csv (341행)


In [12]:
pred_newsgpt = pd.read_csv("/home/dsail/2026-AINews-Detection/notebook/real-world/chosun_predictions.csv")

misclassified = pred_newsgpt[pred_newsgpt["gt"] != pred_newsgpt["pred"]].reset_index(drop=True)
print(f"전체: {len(pred_newsgpt)}  오분류: {len(misclassified)}  정확도: {1 - len(misclassified)/len(pred_newsgpt):.4f}")
misclassified 

전체: 341  오분류: 80  정확도: 0.7654


,title,text,gt,pred,ai_score
0,"[단독] LG, 4000억 들여 국내 첫 ‘로봇 훈련소’ 짓는다","[단독] LG, 4000억 들여 국내 첫 ‘로봇 훈련소’ 짓는다\n\nLG전자가 휴...",1,0,0.758644
1,"삼성 반도체 공장行 레미콘 트럭, 노조 방해에 포기","삼성 반도체 공장行 레미콘 트럭, 노조 방해에 포기\n\n수도권 레미콘 운송노조의 ...",1,0,0.554194
2,"유럽중앙은행, 2년 9개월 만에 금리 0.25%p 인상","유럽중앙은행, 2년 9개월 만에 금리 0.25%p 인상\n\n유럽중앙은행(ECB)이...",1,0,0.617307
3,서울 모인 엔비디아·AWS·제조 대기업… “피지컬 AI 승부처는 현장 데이터”,서울 모인 엔비디아·AWS·제조 대기업… “피지컬 AI 승부처는 현장 데이터”\n\...,1,0,0.798692
4,"LG엔솔, 중국 신왕다 특허전 승리… 라이선스 계약 체결","LG엔솔, 중국 신왕다 특허전 승리… 라이선스 계약 체결\n\nLG에너지솔루션이 중...",1,0,0.783812
...,...,...,...,...,...
75,설 앞두고 고병원성 AI 확산에... 정부 “살처분 농가 산란계 재공급 추진”,설 앞두고 고병원성 AI 확산에... 정부 “살처분 농가 산란계 재공급 추진”\n\n,0,1,0.883169
76,"방통위, 온라인 마약 매매 정보 연간 3만건 적발","방통위, 온라인 마약 매매 정보 연간 3만건 적발\n\n",0,1,0.860763
77,"정부, 상반기 2%대 물가 안착 위해 11조 투입","정부, 상반기 2%대 물가 안착 위해 11조 투입\n\n",0,1,0.871357
78,인간보다 17배 강하다... KAIST의 ‘헤라클레스 인공근육’,인간보다 17배 강하다... KAIST의 ‘헤라클레스 인공근육’\n\n,0,1,0.862900


In [35]:
import pandas as pd
df = pd.read_csv("/home/dsail/2026-AINews-Detection/model/results/2026-06-01/RAID-news-22-28-26-322680/eval_newsgpt/predictions_20260605_121425/newsgpt_ai_news_human_preds.csv")
df

,sample_idx,csv_row,label,label_name,score,pred_at_0.5,pred_at_best,correct_at_0.5,correct_at_best,file,csv_path
0,0,0,1,ai,0.837056,1,1,1,1,newsgpt_ai_news_human.csv,/home/dsail/2026-AINews-Detection/notebook/new...
1,1,1,1,ai,0.819248,1,1,1,1,newsgpt_ai_news_human.csv,/home/dsail/2026-AINews-Detection/notebook/new...
2,2,2,1,ai,0.810677,1,1,1,1,newsgpt_ai_news_human.csv,/home/dsail/2026-AINews-Detection/notebook/new...
3,3,3,1,ai,0.813067,1,1,1,1,newsgpt_ai_news_human.csv,/home/dsail/2026-AINews-Detection/notebook/new...
4,4,4,1,ai,0.802193,1,1,1,1,newsgpt_ai_news_human.csv,/home/dsail/2026-AINews-Detection/notebook/new...
...,...,...,...,...,...,...,...,...,...,...,...
868,868,868,0,human,0.539362,1,0,0,1,newsgpt_ai_news_human.csv,/home/dsail/2026-AINews-Detection/notebook/new...
869,869,869,0,human,0.623205,1,0,0,1,newsgpt_ai_news_human.csv,/home/dsail/2026-AINews-Detection/notebook/new...
870,870,870,0,human,0.719914,1,0,0,1,newsgpt_ai_news_human.csv,/home/dsail/2026-AINews-Detection/notebook/new...
871,871,871,0,human,0.558357,1,0,0,1,newsgpt_ai_news_human.csv,/home/dsail/2026-AINews-Detection/notebook/new...


In [49]:
ai_article = pd.read_csv("/home/dsail/2026-AINews-Detection/notebook/real-world/NewsGPT/newsgpt_ai_news_human.csv")["ai_article"][868]
ai_article

'The villagers said the soldiers came first, firing indiscriminately. Then came civilians, accompanying the soldiers, to loot and burn. Now in Bangladesh, 20 Muslims and Hindus gave interviews in which they recounted how they were forced out of their village of Kha Maung Seik in Myanmar s Rakhine State on Aug. 25.  The military brought some Rakhine Buddhists with them and torched the village,  said Kadil Hussein, 55.  All the Muslims in our village, about 10,000, fled. Some were killed by gunshots, the rest came here. There s not a single person left.  Hussein is staying with hundreds of other new arrivals at the Kutapalong refugee settlement, already home to thousands of Rohingya who fled earlier. Nearly 150,000 Rohingya have arrived in Bangladesh since Aug. 25, when insurgents of the Arakan Rohingya Salvation Army launched attacks on security forces in Rakhine State. Reuters interviewed villagers from Kha Maung Seik and neighboring hamlets, who described killings and the burning of h